# Setup & How To

This course teaches prompt engineering using the Claude API in Python. Each chapter is a Jupyter notebook that you run cell by cell, experimenting with prompts and inspecting model output directly. You'll need an Anthropic API key, which you can get at [console.anthropic.com](https://console.anthropic.com).

## Prerequisites & setup

1. **Install Python 3.10+** from [python.org](https://www.python.org/downloads/) (or use your system / conda Python).
2. **Create a virtual environment and install the dependencies** from the project root:
   ```bash
   python3 -m venv .venv
   source .venv/bin/activate          # Windows: .venv\Scripts\activate
   pip install -r requirements.txt
   ```
3. **Register the environment as a Jupyter kernel** (so notebooks run with the packages you just installed):
   ```bash
   python -m ipykernel install --user --name prompt-eng-python --display-name "Python (prompt-eng)"
   ```
4. **Configure your API key**: copy `.env.example` to `.env` **in the project root** and set `ANTHROPIC_API_KEY`.
5. **Open notebooks** in VS Code (with the Jupyter extension) or `jupyter lab`, then select the **Python (prompt-eng)** kernel for each notebook.

## The helper

All chapters import from `lib/helpers.py`, which wraps the Anthropic Messages API in a single `get_completion(prompt, system="")` function and exports the pinned `MODEL` string, the shared `client`, and a few constants.

The Messages API requires three things: `model`, `max_tokens`, and a `messages` list. The `system` prompt is optional.

Two details about current Claude models shape how the helper works:

* **Sampling parameters.** The course defaults to `claude-opus-5-5`. Current models like this one no longer accept `temperature` (sending it returns a `400`), and the 1.x Python SDK dropped the `temperature=` keyword argument altogether. Older models such as `claude-sonnet-4-6` still honour it, so for those the helper sends `temperature=0` through the SDK's `extra_body` escape hatch to make outputs as repeatable as possible (`SAMPLING_PARAMS` is `{"extra_body": {"temperature": 0}}` on older models and `{}` on current ones). Either way, outputs are not guaranteed to be identical between runs, so the graders check for *properties* of the answer (contains `404`, equals exactly `pytest`, has at least 800 words) rather than exact text. A well-written prompt passes reliably.
* **Thinking.** Current models reason internally before answering. That reasoning comes back as separate `thinking` blocks and counts toward `max_tokens`, which is why the helper uses a generous `MAX_TOKENS = 16000` and why `response_text()` joins only the `text` blocks of a response.

You can switch models without touching code by setting `CLAUDE_MODEL` in `.env` (for example `CLAUDE_MODEL=claude-sonnet-5-5`). Restart the kernel after editing `.env`.

In [1]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS
from lib.helpers import api_key_configured

print("Using model:", MODEL)
print("API key configured:", api_key_configured())  # prints True/False, never the key itself

Using model: claude-opus-5-5
API key configured: True


In [2]:
print(get_completion("In one sentence, what is an idempotent HTTP method?"))

An idempotent HTTP method is one where making the same request multiple times has the same effect on the server as making it once. Examples are GET, PUT, and DELETE, while POST is not idempotent.


`get_completion(prompt, system="")` is the only helper you need for most chapters. It is a thin wrapper around `client.messages.create` defined in [`lib/helpers.py`](../lib/helpers.py). The optional `system` argument defaults to an empty string (no system prompt).

Two conveniences are built in. While an exercise prompt still contains its placeholder (`"[Replace this text]"`), `get_completion` skips the API call, prints a reminder, and returns an empty string, so an unsolved exercise grades `False` without spending tokens. And if Claude ever declines a request (`stop_reason == "refusal"`), the helpers print a warning instead of silently returning an empty answer.

There is no `await` anywhere: `client` is the SDK's synchronous `anthropic.Anthropic` client, so each call simply blocks until Claude responds. (For `asyncio` applications the SDK also ships `anthropic.AsyncAnthropic`, with the same methods as coroutines.)

### Under the hood

Here is what `get_completion` does, written out in full. Notice that `response.content` is a **list of blocks**, each with a `type`. On current models the first block is often a `thinking` block, so always select the `text` blocks instead of reaching for `content[0]`.

In [3]:
response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Name one benefit of type hints in Python, in one sentence."}],
    **SAMPLING_PARAMS,
)

print("Block types:", [block.type for block in response.content])
print("Stop reason:", response.stop_reason)
print("Usage:", response.usage.input_tokens, "input tokens /", response.usage.output_tokens, "output tokens")
print()
print(response_text(response))

Block types: ['text']
Stop reason: end_turn
Usage: 28 input tokens / 31 output tokens

Type hints let static analysis tools like mypy catch type-related bugs before your code runs.
